# CS570 — Project Deliverable 3: Linear Classification Model
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE, NILA KO, KHAING MIN HTWE, YUEXUAN LU  
**Date:** March 25, 2026  
**Instructor:** Dr. Ragnar Lesch | SFBU CS570 — Big Data Processing & Analytics (Spring 2026)

---
**Goal:** Build a Spark MLlib Pipeline (VectorAssembler -> StandardScaler -> Logistic Regression) to predict `high_rating` (Rating >= 4). Evaluate against the D2 naive baseline and interpret the model coefficients.
### Notebook Structure

| Part | Description |
|---|---|
| Part 1 | Pipeline Construction — VectorAssembler, StandardScaler, Logistic Regression |
| Part 2 | Model Evaluation — 5 metrics, baseline comparison, confusion matrix |
| Part 3 | Model Interpretation — coefficients, bar chart, connection to D2 |
| Part 4 | Reflection — 4 written responses |

**Dataset:** MovieLens 1M (ratings.dat · users.dat · movies.dat)  
**Framework:** Apache Spark / PySpark 4.x | Python 3.x

---
## Configuration

**Set `DATA_DIR` to the folder containing `ratings.dat`, `users.dat`, and `movies.dat`.**  


In [1]:
import sys
print(sys.executable)

d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\.venv\Scripts\python.exe


In [2]:
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
# ───────────────────────────────────────────────────────────────

RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.dat')
USERS_PATH   = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH  = os.path.join(DATA_DIR, 'movies.dat')

for path in [RATINGS_PATH, USERS_PATH, MOVIES_PATH]:
    status = 'found' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status}: {path}')

found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\ratings.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\users.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\movies.dat


---
## SparkSession

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D3-Pentanet')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Spark version: 3.5.0


---
## Data Loading

In [4]:
from pyspark.sql.types import(
    StructType, StructField, IntegerType, LongType, StringType,FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])
USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])
MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])

ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_PATH)
users   = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
movies  = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)

print(f'Raings  : {ratings.count():>10}')
print(f'Users   : {users.count():>10}')
print(f'Movies  : {movies.count():>10}')

Raings  :    1000209
Users   :       6040
Movies  :       3883


### Join Tables

In [5]:
joined = (
    ratings
    .join(users, on='UserID', how ='inner')
    .join(movies, on = 'MovieID', how ='inner')
).cache()

print(f'Raws    : {joined.count():,}')
print(f'Columns : {len(joined.columns)}')
joined.show(3)

Raws    : 1,000,209
Columns : 10
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|   1193|     1|   5.0|978300760|     F|  1|        10|  48067|One Flew Over the...|               Drama|
|    661|     1|   3.0|978302109|     F|  1|        10|  48067|James and the Gia...|Animation|Childre...|
|    914|     1|   3.0|978301968|     F|  1|        10|  48067| My Fair Lady (1964)|     Musical|Romance|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
only showing top 3 rows



---
## Feature Engineering

All features are built inline below, replicating the D2 feature set used in EDA.

In [6]:
from pyspark.sql import functions as F 

df = joined

# Target label
df = df.withColumn('high_rating', F.when(F.col('Rating') >=4, 1).otherwise(0))

# Movie-level aggregates
movie_stats = df.groupBy('MovieID').agg(
    F.avg('Rating').alias('movie_avg_rating'),
    F.count('Rating').alias('movie_popularity')
)

df = df.join(movie_stats, on='MovieID', how='left')
df = df.withColumn('log_movie_popularity', F.log(F.col('movie_popularity') + 1))
df = df.withColumn(
    'release_year',
    F.regexp_extract(F.col('Title'), r'\((\d{4})\)', 1).cast('int')
)

df = df.withColumn('movie_age', 2000 - F.col('release_year'))

# user-level aggregates
user_stats = df.groupBy('UserID').agg(
    F.avg('Rating').alias('user_avg_rating'),
    F.count('Rating').alias('user_rating_count')
)
df = df.join(user_stats, on='UserID', how='left')
global_avg = df.agg(F.avg('Rating')).collect()[0][0]
df = df.withColumn('rating_deviation', F.col('user_avg_rating') - global_avg)

# interaction and genre features 

df = df.withColumn('user_movie_interaction', F.col('user_avg_rating') * F.col('movie_avg_rating'))
df = df.withColumn('num_genres', F.size(F.split(F.col('Genres'), r'\|')))
df = df.withColumn('is_action',    F.when(F.col('Genres').contains('Action'),    1).otherwise(0))
df = df.withColumn('is_horror',    F.when(F.col('Genres').contains('Horror'),    1).otherwise(0))
df = df.withColumn('is_war',       F.when(F.col('Genres').contains('War'),       1).otherwise(0))
df = df.withColumn('is_film_noir', F.when(F.col('Genres').contains('Film-Noir'), 1).otherwise(0))

# Fill nulls from release_year regex failures
df = df.na.fill(0, subset=['release_year', 'movie_age'])

df = df.cache()
print(f'Rows : {df.count():,}  |  Cols : {len(df.columns)}')

Rows : 1,000,209  |  Cols : 25


---
## Pre-Modeling Checklist

Four conditions verified before training. All must pass.

### Check 1 - Null Audit

`VectorAssembler` silently propagates NaN: a single null corrupts the entire feature vector without raising an error.

In [7]:
null_counts = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

total_nulls = sum(null_counts.values())
if total_nulls == 0:
    print('PASS -- Total nulls: 0.  All columns are clean.')
else:
    print('FAIL --', {k: v for k, v in null_counts.items() if v > 0})

PASS -- Total nulls: 0.  All columns are clean.


### Check 2 - Target Class Balance

Accuracy is misleading for imbalanced targets. We store `majority_rate` for the baseline table in Part 2.

In [8]:
total_rows = df.count()
(
    df.groupBy('high_rating')
      .count()
      .withColumn('pct', F.round(F.col('count') / total_rows * 100, 2))
      .orderBy('high_rating')
).show()

majority_rate = df.filter(F.col('high_rating') == 1).count() / total_rows
print(f'Majority class prevalence : {majority_rate:.4f}')
print(f'Naive baseline accuracy   : {majority_rate:.4f}  (always predict high_rating=1)')

+-----------+------+-----+
|high_rating| count|  pct|
+-----------+------+-----+
|          0|424928|42.48|
|          1|575281|57.52|
+-----------+------+-----+

Majority class prevalence : 0.5752
Naive baseline accuracy   : 0.5752  (always predict high_rating=1)


### Check 3 - Feature Types + Gender Encoding

MLlib requires numeric inputs. `Gender` (M/F string) is encoded to a binary integer (M=1, F=0).

In [9]:
df = df.withColumn('gender_encoded', F.when(F.col('Gender') == 'M', 1).otherwise(0))

FEATURE_COLS = [
    'user_movie_interaction',  # D2 cor 0.484 - strongest predictor
    'movie_avg_rating',        # D2 cor 0.410
    'user_avg_rating',         # D2 cor 0.338
    'log_movie_popularity',    # D2 cor 0.212 (log-transformed)
    'movie_age',               # D2 cor 0.131
    'Age',                     # user age code 1/18/25/35/45/50/56
    'gender_encoded',          # binary M=1, F=0
    'num_genres',              # count of genres
    'is_action',               # genre binary flags
    'is_horror',
    'is_war',
    'is_film_noir',
]
# NOTE: 'high_rating' is the TARGET -- it must NOT appear in FEATURE_COLS

NUMERIC_TYPES = {'int', 'bigint', 'double', 'float', 'long'}
schema_map = dict(df.dtypes)
all_ok = True
for feat in FEATURE_COLS:
    dtype = schema_map.get(feat, 'MISSING')
    ok = dtype in NUMERIC_TYPES
    if not ok:
        all_ok = False
    print(f"  {'PASS' if ok else 'FAIL'}  {feat:<28s}  {dtype}")
print()
print('All features numeric:', all_ok)

  PASS  user_movie_interaction        double
  PASS  movie_avg_rating              double
  PASS  user_avg_rating               double
  PASS  log_movie_popularity          double
  PASS  movie_age                     int
  PASS  Age                           int
  PASS  gender_encoded                int
  PASS  num_genres                    int
  PASS  is_action                     int
  PASS  is_horror                     int
  PASS  is_war                        int
  PASS  is_film_noir                  int

All features numeric: True


### Check 4 - Outlier Check

`movie_popularity` ranges 1 to ~3,428 while other features range 0-8.  
We use `log_movie_popularity` to compress this skew. `StandardScaler` normalises all features to unit variance.

In [10]:
df.select(
    F.min('movie_popularity').alias('pop_min'),
    F.max('movie_popularity').alias('pop_max'),
    F.round(F.mean('movie_popularity'), 1).alias('pop_mean'),
    F.round(F.min('log_movie_popularity'), 3).alias('log_min'),
    F.round(F.max('log_movie_popularity'), 3).alias('log_max'),
    F.round(F.mean('log_movie_popularity'), 3).alias('log_mean'),
).show()
print('Decision: use log_movie_popularity (range ~0-8) not raw popularity (1-3428).')

+-------+-------+--------+-------+-------+--------+
|pop_min|pop_max|pop_mean|log_min|log_max|log_mean|
+-------+-------+--------+-------+-------+--------+
|      1|   3428|   816.2|  0.693|   8.14|   6.324|
+-------+-------+--------+-------+-------+--------+

Decision: use log_movie_popularity (range ~0-8) not raw popularity (1-3428).


### Train / Test Split - 80/20, seed=42

The same `seed=42` must be reused in D4 so the comparison is fair.

In [11]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
train = train.cache()
test  = test.cache()
print(f'Train : {train.count():,}')
print(f'Test  : {test.count():,}')

Train : 800,092
Test  : 200,117


### 1a. Feature Assembly

**Feature set and rationale** (based on D2 feature-target correlation summary):

| Feature | D2 Correlation | Rationale |
|---|---|---|
| `user_movie_interaction` | 0.484 | Strongest signal: product of user and movie averages captures joint quality |
| `movie_avg_rating` | 0.410 | Community wisdom -- highly-rated movies attract high individual ratings |
| `user_avg_rating` | 0.338 | Leniency bias: generous raters rate everything highly |
| `log_movie_popularity` | 0.212 | Popular movies tend to be higher quality; log compresses outlier skew |
| `movie_age` | 0.131 | Older surviving films tend to be classics rated highly |
| `Age` | n/a | User age code (1/18/25/35/45/50/56) -- demographic signal |
| `gender_encoded` | n/a | Binary (M=1, F=0) -- tests gender effect on rating |
| `num_genres` | -0.004 | Weak but included; multi-genre films have broader appeal |
| `is_action` | n/a | Genre binary -- action films show distinct rating patterns |
| `is_horror` | n/a | Genre binary -- horror fans rate genre films distinctly |
| `is_war` | n/a | Genre binary -- war films attract serious, high-rating viewers |
| `is_film_noir` | n/a | Genre binary -- niche genre with dedicated high-rating fans |

**Excluded:**
- `high_rating` -- the target itself; including it is data leakage 
- `rating_deviation` -- perfectly collinear with `user_avg_rating` (deviation = avg - global_avg)
- `movie_popularity` -- replaced by `log_movie_popularity` to handle skew
- `release_year` -- redundant with `movie_age`
- `primary_genre`, `movie_era` -- categorical strings needing StringIndexer+OHE (deferred to D4)

In [12]:
from pyspark.ml.feature        import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml                import Pipeline

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol='raw_features')

print(f'Feature count : {len(FEATURE_COLS)}')
print('Features      :', FEATURE_COLS)

Feature count : 12
Features      : ['user_movie_interaction', 'movie_avg_rating', 'user_avg_rating', 'log_movie_popularity', 'movie_age', 'Age', 'gender_encoded', 'num_genres', 'is_action', 'is_horror', 'is_war', 'is_film_noir']


### 1b. Feature Scaling 

**Why feature scaling matters for Logistic Regression but not for tree-based models:**

Logistic Regression learns by gradient descent. It adjusts every weight using the same learning rate. If `log_movie_popularity` ranges 0-8 while `is_action` is 0/1, the large-scale feature dominates every gradient update. The model spends hundreds of iterations correcting that one large weight while others barely move -- slow and suboptimal convergence. `StandardScaler` brings all features to unit variance, giving gradient descent a balanced loss surface where each feature is updated at a comparable rate.

Tree-based models (Random Forest, GBT) split by threshold comparisons like `log_movie_popularity > 4.5`. The absolute scale does not affect which threshold is chosen -- a tree finds the optimal split equally well on raw or standardised values. Trees are invariant to monotonic feature transformations; scaling has no effect. *(Week 9A: 'scaling is a prerequisite for gradient-based learners; trees do not care.')*

`withMean=False` is required: `VectorAssembler` produces a sparse vector, and centering a sparse vector densifies it, removing the memory benefit of sparsity.

In [13]:
scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withStd=True,
    withMean=False
)
print('StandardScaler: withStd=True, withMean=False')

StandardScaler: withStd=True, withMean=False


### 1c. Build the Pipeline

All three stages -- `VectorAssembler -> StandardScaler -> LogisticRegression` -- are chained into one `Pipeline`.

**Key benefit:** fitting on `train` means the scaler learns mean/std only from training data. Fitting on the full dataset would let test statistics influence the scaler -- a form of data leakage that inflates metrics.

In [14]:
lr = LogisticRegression(featuresCol='features', labelCol='high_rating', maxIter=100)

pipeline = Pipeline(stages=[assembler, scaler, lr])

print('Fitting pipeline on training data ...')
model = pipeline.fit(train)
print('Pipeline fir complete')

predictions = model.transform(test)
print(f'Predictions: {predictions.count():,} rows')
predictions.select('high_rating', 'prediction', 'probability').show(5, truncate=False)

Fitting pipeline on training data ...
Pipeline fir complete
Predictions: 200,117 rows
+-----------+----------+----------------------------------------+
|high_rating|prediction|probability                             |
+-----------+----------+----------------------------------------+
|1          |1.0       |[0.09052857739430697,0.909471422605693] |
|1          |1.0       |[0.14232233274088346,0.8576776672591165]|
|1          |1.0       |[0.12197449520243492,0.8780255047975651]|
|1          |1.0       |[0.2746541533950112,0.7253458466049888] |
|1          |1.0       |[0.16710623671281646,0.8328937632871836]|
+-----------+----------+----------------------------------------+
only showing top 5 rows



---
## Part 2: Model Evaluation

### 2a. Compute Metrics

All five required metrics: **AUC-PR**, **Accuracy**, **Weighted Precision**, **Weighted Recall**, **F1**.

In [15]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol='high_rating', rawPredictionCol='rawPrediction', metricName='areaUnderPR'
)

auc_pr = auc_eval.evaluate(predictions)

mc_eval = MulticlassClassificationEvaluator(labelCol='high_rating', predictionCol='prediction')
lr_metrics = {}

for m in ['accuracy', 'weightedPrecision', 'weightedRecall', 'f1']:
    mc_eval.setMetricName(m)
    lr_metrics[m] = mc_eval.evaluate(predictions)

print('=' * 44)
print(f"  AUC-PR             : {auc_pr:.4f}")
print(f"  Accuracy           : {lr_metrics['accuracy']:.4f}")
print(f"  Weighted Precision : {lr_metrics['weightedPrecision']:.4f}")
print(f"  Weighted Recall    : {lr_metrics['weightedRecall']:.4f}")
print(f"  F1 (weighted)      : {lr_metrics['f1']:.4f}")
print('=' * 44)

  AUC-PR             : 0.8150
  Accuracy           : 0.7215
  Weighted Precision : 0.7203
  Weighted Recall    : 0.7215
  F1 (weighted)      : 0.7165


### 2b. Baseline Comparison 

The naive baseline always predicts the majority class (`high_rating=1`). Accuracy = prevalence. Recall = 1.0. AUC-PR = prevalence (zero ranking ability).

**Why is AUC-PR equal to prevalence for the baseline?**  
AUC-PR measures whether the model ranks positives above negatives. A baseline assigning the same score to every instance cannot rank -- the probability any random sample is positive is just the prevalence. Precision never rises above that floor: PR curve is a flat line at P = prevalence. The gap between baseline AUC-PR and the model's AUC-PR is the evidence of learning.

In [16]:
import pandas as pd

train_dist = train.groupBy('high_rating').count().toPandas()
train_total = train_dist['count'].sum()
trian_maj = train_dist.loc[train_dist['high_rating'] == 1, 'count'].values[0]
train_prev = trian_maj / train_total
baseline_f1 = 2*train_prev / (train_prev + 1.0)

print('Training class distribution:')
print(train_dist.to_string(index=False))
print()

comp = pd.DataFrame({
    'Metric': ['AUC-PR','Accuracy','Weighted Precision','Weighted Recall','F1 (weighted)'],
    'Naive Baseline (always predict 1)': [
        f'{train_prev:.4f}  (= prevalence)',
        f'{train_prev:.4f}',
        f'{train_prev:.4f}',
        '1.0000',
        f'{baseline_f1:.4f}  (inflated by perfect recall)'
    ],
    'Logistic Regression': [
        f'{auc_pr:.4f}',
        f"{lr_metrics['accuracy']:.4f}",
        f"{lr_metrics['weightedPrecision']:.4f}",
        f"{lr_metrics['weightedRecall']:.4f}",
        f"{lr_metrics['f1']:.4f}"
    ]
})
print(comp.to_string(index=False))

Training class distribution:
 high_rating  count
           1 460523
           0 339569

            Metric    Naive Baseline (always predict 1) Logistic Regression
            AUC-PR               0.5756  (= prevalence)              0.8150
          Accuracy                               0.5756              0.7215
Weighted Precision                               0.5756              0.7203
   Weighted Recall                               1.0000              0.7215
     F1 (weighted) 0.7306  (inflated by perfect recall)              0.7165


**Does the model beat the baseline? (Actual results)**

The critical metric is **AUC-PR**:

| Metric | Naive Baseline | Logistic Regression | Gain |
|---|---|---|---|
| AUC-PR | 0.5756 (= prevalence) | **0.8150** | **+0.2394 (+41.6%)** |
| Accuracy | 0.5756 | 0.7215 | +0.1459 |
| F1 (weighted) | 0.7306 *(inflated)* | 0.7165 | –0.0141 |

**AUC-PR is the most honest measure here.** The naive baseline AUC-PR of 0.5756 represents pure chance (the probability a random sample is positive). Logistic Regression's AUC-PR of 0.8150 is an absolute gain of **+0.2394** — the model has genuine ranking ability.

The baseline F1 (0.7306) is higher than LR's F1 (0.7165) because the baseline achieves perfect recall (1.0) by always predicting the majority class, which inflates F1 arithmetically. This is misleading — the baseline never correctly identifies a negative. We treat **AUC-PR as the primary metric** throughout D3 and D4.

### 2c. Confusion Matrix

In [17]:
print('Confusion Matrix (actual = rows | predicted = cols)\n')
cm = predictions.groupBy('high_rating', 'prediction').count().orderBy('high_rating', 'prediction')
cm.show()

cm_dict = {(int(r['high_rating']), int(r['prediction'])) : r['count'] for r in cm.collect()}
TP = cm_dict.get((1,1), 0)
TN = cm_dict.get((0,0), 0)
FP = cm_dict.get((0,1), 0)
FN = cm_dict.get((1,0), 0)
print(f'True  Positives (TP) : {TP:>8,}   correctly predicted high rating')
print(f'True  Negatives (TN) : {TN:>8,}   correctly predicted low rating')
print(f'False Positives (FP) : {FP:>8,}   recommended a movie the user will NOT enjoy')
print(f'False Negatives (FN) : {FN:>8,}   missed a movie the user WOULD have enjoyed')

Confusion Matrix (actual = rows | predicted = cols)

+-----------+----------+-----+
|high_rating|prediction|count|
+-----------+----------+-----+
|          0|       0.0|49815|
|          0|       1.0|35544|
|          1|       0.0|20189|
|          1|       1.0|94569|
+-----------+----------+-----+

True  Positives (TP) :   94,569   correctly predicted high rating
True  Negatives (TN) :   49,815   correctly predicted low rating
False Positives (FP) :   35,544   recommended a movie the user will NOT enjoy
False Negatives (FN) :   20,189   missed a movie the user WOULD have enjoyed


**Confusion matrix interpretation (actual counts):**

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN = 49,815 | FP = 35,544 |
| **Actual 1** | FN = 20,189 | TP = 94,569 |

The model makes **35,544 False Positives** (recommends a movie the user will NOT enjoy) versus **20,189 False Negatives** (misses a movie the user would have loved). FP count is 76% higher than FN — the model errs on the side of over-recommending.

For a recommendation system, **False Positives are the worse error**:
- A FP is directly experienced: the user watches a disappointing film and loses trust in the recommender.
- A FN is invisible: the user simply never sees a movie they might have liked — no harm felt.

In D4 we will raise the classification threshold above 0.5 to favour precision (reduce FP) at the cost of more FN, improving user experience. The current AUC-PR of 0.8150 shows the model has strong ranking ability; the threshold is a separate dial.

---
### 2d. Threshold Tuning

The default classification threshold is **0.5**: predict `high_rating=1` whenever P(1) ≥ 0.5. This is arbitrary — it treats every False Positive and every False Negative as equally costly, but the confusion matrix in Part 2c shows they are not equal for a recommendation system.

Raising the threshold shifts the model toward **precision** (fewer bad recommendations shown to users) at the cost of **recall** (more good films never surfaced):

- **Higher threshold** → fewer FP (better user experience), more FN (some good films hidden)
- **Lower threshold** → more FP (disappointing recommendations), fewer FN (broader coverage)

We sweep thresholds **0.30 → 0.80** (step 0.05), re-apply each to the existing `predictions` DataFrame (no retraining), and record Precision, Recall, F1, and FP / FN counts at each point.

In [19]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
import pandas as pd

# vector_to_array converts VectorUDT → ArrayType(DoubleType) so [1] indexing works
prob_col   = vector_to_array(F.col('probability'))[1]
thresholds = [round(0.30 + i * 0.05, 2) for i in range(11)]   # 0.30 → 0.80

thresh_results = []
for thresh in thresholds:
    pred_t = predictions.withColumn(
        'pred_t', F.when(prob_col >= thresh, 1.0).otherwise(0.0)
    )
    cm_t = {
        (int(r['high_rating']), int(r['pred_t'])): r['count']
        for r in pred_t.groupBy('high_rating', 'pred_t').count().collect()
    }
    TP_t = cm_t.get((1, 1), 0)
    TN_t = cm_t.get((0, 0), 0)
    FP_t = cm_t.get((0, 1), 0)
    FN_t = cm_t.get((1, 0), 0)

    prec = TP_t / (TP_t + FP_t) if (TP_t + FP_t) > 0 else 0.0
    rec  = TP_t / (TP_t + FN_t) if (TP_t + FN_t) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    acc  = (TP_t + TN_t) / (TP_t + TN_t + FP_t + FN_t)

    thresh_results.append({
        'threshold': thresh,
        'precision': round(prec, 4),
        'recall':    round(rec,  4),
        'f1':        round(f1,   4),
        'accuracy':  round(acc,  4),
        'FP':        FP_t,
        'FN':        FN_t,
    })

thresh_df = pd.DataFrame(thresh_results)
best_row  = thresh_df.loc[thresh_df['f1'].idxmax()]

print(thresh_df.to_string(index=False))
print(f"\nBest F1  → threshold={best_row['threshold']}  "
      f"F1={best_row['f1']:.4f}  Prec={best_row['precision']:.4f}  Rec={best_row['recall']:.4f}  "
      f"FP={int(best_row['FP']):,}  FN={int(best_row['FN']):,}")

 threshold  precision  recall     f1  accuracy    FP    FN
      0.30     0.6578  0.9477 0.7766    0.6873 56577  6001
      0.35     0.6742  0.9263 0.7804    0.7010 51374  8461
      0.40     0.6911  0.8993 0.7816    0.7118 46121 11558
      0.45     0.7082  0.8659 0.7792    0.7185 40937 15387
      0.50     0.7268  0.8241 0.7724    0.7215 35544 20189
      0.55     0.7462  0.7743 0.7600    0.7195 30227 25897
      0.60     0.7666  0.7122 0.7384    0.7106 24884 33023
      0.65     0.7889  0.6386 0.7058    0.6947 19611 41479
      0.70     0.8122  0.5490 0.6551    0.6686 14568 51759
      0.75     0.8370  0.4438 0.5800    0.6315  9915 63834
      0.80     0.8654  0.3222 0.4695    0.5826  5750 77788

Best F1  → threshold=0.4  F1=0.7816  Prec=0.6911  Rec=0.8993  FP=46,121  FN=11,558


In [20]:
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=UserWarning)
plt.switch_backend('agg')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Precision / Recall / F1 vs Threshold
ax1 = axes[0]
ax1.plot(thresh_df['threshold'], thresh_df['precision'], 'b-o', markersize=5, label='Precision')
ax1.plot(thresh_df['threshold'], thresh_df['recall'],    'r-o', markersize=5, label='Recall')
ax1.plot(thresh_df['threshold'], thresh_df['f1'],        'g-o', markersize=5, label='F1')
ax1.axvline(best_row['threshold'], color='gray', linestyle='--', linewidth=1,
            label=f"Best F1 @ {best_row['threshold']}")
ax1.set_xlabel('Classification Threshold')
ax1.set_ylabel('Score')
ax1.set_title('Precision / Recall / F1 vs Threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: FP / FN trade-off vs Threshold
ax2 = axes[1]
ax2.plot(thresh_df['threshold'], thresh_df['FP'], 'r-o', markersize=5, label='False Positives (bad recs)')
ax2.plot(thresh_df['threshold'], thresh_df['FN'], 'b-o', markersize=5, label='False Negatives (missed)')
ax2.axvline(0.5,                  color='gray',  linestyle=':',  linewidth=1, label='Default 0.5')
ax2.axvline(best_row['threshold'], color='green', linestyle='--', linewidth=1,
            label=f"Best F1 @ {best_row['threshold']}")
ax2.set_xlabel('Classification Threshold')
ax2.set_ylabel('Count')
ax2.set_title('FP / FN Trade-off vs Threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.path.abspath(''), 'threshold_tuning.png'), dpi=120)
plt.show()
print('Chart saved: threshold_tuning.png')

Chart saved: threshold_tuning.png


---
### 2e. Hyperparameter Tuning

Logistic Regression has two key regularization hyperparameters:

| Parameter | Effect |
|---|---|
| `regParam` (λ) | Penalty strength — larger values shrink all coefficients toward zero, reducing overfitting at the cost of bias |
| `elasticNetParam` (α) | Mix of L1 (α=1.0, drives weak coefficients to exactly zero) and L2 (α=0.0, shrinks all evenly but never zeroes) |

We use **`TrainValidationSplit`** (single 80/20 internal split) over a 3 × 2 grid — 6 combinations, 6 model fits. `CrossValidator` (k-fold) is more statistically thorough but k× slower on 800 K training rows; `TrainValidationSplit` is the practical choice for this dataset. Evaluation metric: **AUC-PR** (consistent with Part 2a — the most honest metric for this class distribution).

In [21]:
from pyspark.ml.tuning      import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation  import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression
from pyspark.ml             import Pipeline
from pyspark.ml.evaluation  import MulticlassClassificationEvaluator
import pandas as pd

lr_tune       = LogisticRegression(featuresCol='features', labelCol='high_rating', maxIter=100)
pipeline_tune = Pipeline(stages=[assembler, scaler, lr_tune])

param_grid = (
    ParamGridBuilder()
    .addGrid(lr_tune.regParam,        [0.0, 0.01, 0.1])
    .addGrid(lr_tune.elasticNetParam, [0.0, 0.5])
    .build()
)

auc_pr_eval = BinaryClassificationEvaluator(
    labelCol='high_rating', rawPredictionCol='rawPrediction', metricName='areaUnderPR'
)

tvs = TrainValidationSplit(
    estimator=pipeline_tune,
    estimatorParamMaps=param_grid,
    evaluator=auc_pr_eval,
    trainRatio=0.8,
    seed=42
)

print(f'Fitting {len(param_grid)} combinations via TrainValidationSplit ...')
tvs_model     = tvs.fit(train)
best_pipeline = tvs_model.bestModel
best_lr_model = best_pipeline.stages[-1]

best_reg   = best_lr_model.getRegParam()
best_alpha = best_lr_model.getElasticNetParam()
print(f'Best regParam       : {best_reg}')
print(f'Best elasticNetParam: {best_alpha}')

# Evaluate tuned model on test set
best_preds  = best_pipeline.transform(test)
best_auc_pr = auc_pr_eval.evaluate(best_preds)

mc_tune = MulticlassClassificationEvaluator(labelCol='high_rating', predictionCol='prediction')
tuned_metrics = {}
for m in ['accuracy', 'weightedPrecision', 'weightedRecall', 'f1']:
    mc_tune.setMetricName(m)
    tuned_metrics[m] = mc_tune.evaluate(best_preds)

print()
comp_tune = pd.DataFrame({
    'Metric': ['AUC-PR', 'Accuracy', 'Weighted Precision', 'Weighted Recall', 'F1'],
    'Default LR (regParam=0.0, alpha=0.0)': [
        f'{auc_pr:.4f}',
        f"{lr_metrics['accuracy']:.4f}",
        f"{lr_metrics['weightedPrecision']:.4f}",
        f"{lr_metrics['weightedRecall']:.4f}",
        f"{lr_metrics['f1']:.4f}",
    ],
    f'Tuned LR (regParam={best_reg}, alpha={best_alpha})': [
        f'{best_auc_pr:.4f}',
        f"{tuned_metrics['accuracy']:.4f}",
        f"{tuned_metrics['weightedPrecision']:.4f}",
        f"{tuned_metrics['weightedRecall']:.4f}",
        f"{tuned_metrics['f1']:.4f}",
    ]
})
print(comp_tune.to_string(index=False))

Fitting 6 combinations via TrainValidationSplit ...
Best regParam       : 0.0
Best elasticNetParam: 0.5

            Metric Default LR (regParam=0.0, alpha=0.0) Tuned LR (regParam=0.0, alpha=0.5)
            AUC-PR                               0.8150                             0.8150
          Accuracy                               0.7215                             0.7215
Weighted Precision                               0.7203                             0.7203
   Weighted Recall                               0.7215                             0.7215
                F1                               0.7165                             0.7165


---
## Part 3: Model Interpretation

### 3a. Coefficient Analysis 

Coefficients extracted from the fitted pipeline and ranked by absolute magnitude.

In [22]:
import pandas as pd

lr_model = model.stages[-1]

coeff_df = pd.DataFrame({
    'feature' :    FEATURE_COLS,
    'coefficient': lr_model.coefficients.toArray()
})
coeff_df['abs_coeff'] = coeff_df['coefficient'].abs()
coeff_df = coeff_df.sort_values('abs_coeff', ascending=False).reset_index(drop=True)

print(f'Intercept: {lr_model.intercept:.4f}\n')
print(coeff_df.to_string(index=False))

Intercept: -15.7007

               feature  coefficient  abs_coeff
      movie_avg_rating     1.610142   1.610142
       user_avg_rating     1.234588   1.234588
user_movie_interaction    -0.895680   0.895680
                   Age    -0.089500   0.089500
  log_movie_popularity    -0.029834   0.029834
             is_horror     0.029414   0.029414
             movie_age    -0.021931   0.021931
            num_genres    -0.018332   0.018332
        gender_encoded     0.014661   0.014661
          is_film_noir     0.010432   0.010432
                is_war     0.010192   0.010192
             is_action     0.009850   0.009850


**Top 3 feature interpretation :**

**1. `movie_avg_rating` — coefficient +1.610 (largest positive)**  
Movies with high community average ratings are the strongest predictor of an individual high rating. When a movie is broadly loved, the model is highly confident the target user will also rate it highly. This matches D2 EDA (correlation 0.410, 2nd overall) and is intuitive: community wisdom is a powerful signal.

**2. `user_avg_rating` — coefficient +1.235 (second largest)**  
Captures leniency bias: users who rate generously on average are likely to give any given movie a high rating. D2 ranked this 3rd (correlation 0.338). The model correctly learns that *who* is rating matters as much as *what* is being rated.

**3. `user_movie_interaction` — coefficient –0.896 (NEGATIVE despite D2 rank #1)**  
This is a surprise: D2 ranked `user_movie_interaction` as the strongest Pearson predictor (|r| = 0.484), yet the LR coefficient is **negative**. The explanation is **multicollinearity**: `user_movie_interaction` is the product of `user_avg_rating` and `movie_avg_rating`, both of which are already in the model. Once those two component features explain their own variance, the interaction term becomes a *suppressor variable* — the model subtracts it to avoid double-counting the joint signal. This is a textbook case of bivariate correlation not translating to multivariate regression coefficients.

**Other notes:**  
- `num_genres` (coeff –0.018) confirms D2's near-zero Pearson correlation (–0.004) — candidate for removal in D4.  
- Genre binary flags (`is_action`, `is_war`, `is_film_noir`, `is_horror`) all carry near-zero but positive coefficients, consistent with their niche-audience, high-rating fan bases.  
- `Age` (coeff –0.090) is negative: older age-code users rate slightly more critically, a new finding not visible in D2's aggregate correlations.

### 3b. Feature Importance Visualization 

In [23]:
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=UserWarning)  # suppress non-interactive backend warning
plt.switch_backend('agg')

colors = ['#27ae60' if c > 0 else '#e74c3c' for c in coeff_df['coefficient']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(coeff_df['feature'], coeff_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Standardised Coefficient Value')
ax.set_title(
    'Logistic Regression Feature Coefficients\n'
    '(green = toward high_rating=1  |  red = toward high_rating=0)'
)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(os.path.abspath(''), 'lr_coefficients.png'), dpi=120)
plt.show()
print('Chart saved: lr_coefficients.png')

Chart saved: lr_coefficients.png


### 3c. Connection to D2 

**1. Did the strongest predictor match the D2 prediction?**  
**No — this is the key finding.** D2 ranked `user_movie_interaction` as the strongest Pearson predictor (|r| = 0.484), so we expected it to have the largest LR coefficient. Instead:
- LR rank #1: `movie_avg_rating` (coeff +1.610)
- LR rank #2: `user_avg_rating` (coeff +1.235)
- LR rank #3: `user_movie_interaction` (coeff **–0.896, negative**)

**Why the discrepancy?** Pearson correlation is *bivariate* — it measures the pairwise linear relationship between one feature and the target, ignoring all other features. Logistic Regression is *multivariate* — it fits all 12 features simultaneously, and each coefficient represents the partial effect of one feature *after controlling for all others*.

`user_movie_interaction = user_avg_rating * movie_avg_rating`. When both component features are already in the model and explaining their own variance, the interaction term becomes redundant. The model assigns it a **negative coefficient** (suppressor variable effect) to avoid double-counting the signal already captured by the two component features. This is a textbook difference between bivariate and multivariate statistics.

**2. Were there other surprises?**  
- `num_genres` had near-zero Pearson correlation in D2 (–0.004) and carries near-zero weight here (–0.018) — confirms it adds only noise.  
- `Age` (–0.090) was not in D2's numeric correlation table; its negative coefficient reveals that older-coded users rate slightly more critically.  
- All genre binary flags are positive (is_horror +0.029, is_film_noir +0.010, is_war +0.010, is_action +0.010) — niche genres attract dedicated fans who rate them highly.

**3. Feature to drop and add for D4:**  
- **Drop** `num_genres` — near-zero in both D2 and D3.  
- **Drop** `user_movie_interaction` or keep with caution — its negative suppressor coefficient is confusing and adds no clean signal once components are present.  
- **Add** one-hot encoding of `primary_genre` via `StringIndexer + OneHotEncoder` — 17 granular genre signals instead of 4 binary flags.  
- **Test** one-hot encoding of `Age` buckets to capture the non-monotone age-rating relationship.

---
## Part 4: Reflection 

**1. Model performance (actual results)**

The most meaningful improvement over the naive baseline is in **AUC-PR**:
- Naive baseline AUC-PR: **0.5756** (= class prevalence, zero ranking ability)
- Logistic Regression AUC-PR: **0.8150**
- Absolute gain: **+0.2394** (+41.6% relative)

The model has genuine ranking ability: it pushes high-rating predictions to higher probabilities and low-rating predictions to lower probabilities across 200,117 test records. Raw accuracy improved from 57.6% to 72.2% (+14.6 pp), but accuracy is less informative because the majority class already gives a high floor.

The baseline F1 (0.7306) appears higher than LR F1 (0.7165). This is not a regression — the baseline achieves perfect recall (1.0) by never predicting class 0, which inflates F1 arithmetically. It correctly classifies 0% of negatives; the model correctly classifies TN = 49,815 negatives.

On the confusion matrix: the model makes **35,544 False Positives** vs **20,189 False Negatives** — it errs toward over-recommending. For a recommendation system **False Positives are the bigger problem**: a user who watches a disappointing film loses trust immediately. A False Negative (a missed good film) is invisible. In D4 we will raise the classification threshold above 0.5 to reduce FP at a cost of more FN.

**2. Linearity limitation**

Logistic Regression models log-odds as a linear combination: w1*x1 + w2*x2 + ... + wn*xn. It assigns one global weight per feature and cannot capture interactions without explicit engineering.

A plausible interaction this model misses is **genre x age group**: Action films may rate highly for younger users (Age code 18-25) but lower for older users (Age code 50-56), while Drama films show the reverse. Logistic Regression assigns one weight to `is_action` and one to `Age` independently -- it cannot learn that the effect of genre changes depending on the viewer's age. In D4 we can address this by (a) explicitly adding an `is_action * Age` interaction feature, or (b) using Gradient Boosted Trees (GBT), which discover joint split conditions automatically.

**3. What would improve the model?**

Two concrete D4 improvements:

1. **Switch to Gradient Boosted Trees (GBT):** GBT captures nonlinear relationships and feature interactions without explicit engineering. Rating behavior is nonlinear -- a movie with movie_avg_rating=4.5 and user_avg_rating=4.0 should nearly guarantee a high rating, while both at 3.5 is uncertain. A linear model cannot express this; GBT can.

2. **Add `primary_genre` as a one-hot encoded feature:** In D3 we excluded `primary_genre` because it is a string. In D4 we will add `StringIndexer -> OneHotEncoder` stages to the pipeline, giving 17+ granular genre signals instead of the 4 binary flags we currently use.

**4. Feature design choice: Age encoding**

MovieLens provides `Age` as seven discrete codes: 1, 18, 25, 35, 45, 50, 56. We treat these as a **single numeric feature**, assuming a linear (monotone) relationship between age and the log-odds of a high rating. This is parsimonious (one degree of freedom) and reasonable if older users rate systematically more critically or generously.

The alternative is **one-hot encoding each age bucket** (7 binary columns, 6 effective degrees of freedom). This makes no linearity assumption and lets the model learn non-monotone age effects. With 1 million rows, six extra parameters cost almost nothing statistically. We plan to test one-hot encoding of `Age` in D4 to check whether the non-parametric representation improves AUC-PR.

---
## Contribution Statement

### Pair A — Part 1 (Pipeline) + Part 2a–2b (Metrics + Baseline)

**AZATBEK ISMAILOV:**  
Computed all five evaluation metrics (AUC-PR = 0.8150, Accuracy = 0.7215, Weighted Precision = 0.7203, Weighted Recall = 0.7215, F1 = 0.7165) using `BinaryClassificationEvaluator` and `MulticlassClassificationEvaluator`. Constructed the baseline comparison table showing the model's AUC-PR gain of +0.2394 over the naive majority-class predictor, and explained why the baseline F1 (0.7306) is misleading — inflated by perfect recall that never identifies any negative. Learned why AUC-PR equals class prevalence for a constant-score baseline and why it is the most honest metric for imbalanced classification.

**FSEHAYE MEDHANIE:**  
Built the MLlib Pipeline skeleton (VectorAssembler → StandardScaler → LogisticRegression) and wrote the pre-modeling checklist (null audit, class balance, feature type check, outlier analysis). Designed the feature set of 12 predictors, documented the rationale for every inclusion and exclusion (data leakage, collinearity, string type), and ran the 80/20 train/test split with seed=42 to ensure reproducibility in D4. Learned that `StandardScaler withMean=False` is required when the input is a sparse vector produced by `VectorAssembler`, and that fitting the scaler only on training data is essential to prevent leakage of test statistics.

---

### Pair B — Part 2c (Confusion Matrix) + Part 3 (Interpretation) + Part 4 (Reflection)

**NILA KO:**  
Extracted and reported the confusion matrix from the test-set predictions (TP = 94,569, TN = 49,815, FP = 35,544, FN = 20,189). Analyzed the practical meaning of each cell: False Positives (bad recommendations shown to users, visible trust damage) versus False Negatives (good films hidden from users, invisible cost). Concluded that FPs are the worse error for a recommendation system and proposed raising the classification threshold in D4 to shift the model toward precision. Learned how the 0.5 default threshold is arbitrary and how the threshold dial independently controls the FP/FN trade-off without changing AUC-PR.

**KHAING MIN HTWE:**  
Performed the coefficient analysis in Part 3a, extracting standardized coefficients from the fitted `LogisticRegressionModel` and ranking by absolute magnitude. Identified the main finding: `user_movie_interaction` carries a negative coefficient (–0.896) despite being D2's top Pearson predictor (|r| = 0.484). Explained this through the lens of multicollinearity — the interaction is the product of `user_avg_rating` and `movie_avg_rating`, both already in the model, making it a suppressor variable. Produced the horizontal bar chart in Part 3b. Learned the critical distinction between bivariate Pearson correlation and partial regression coefficients in a multivariate model.

**YUEXUAN LU:**  
Wrote the D2 connection analysis (Part 3c) and all four Reflection sections (Part 4). Connected the LR coefficient rankings back to D2 EDA, explaining why `movie_avg_rating` (LR rank #1, coeff +1.610) displaced `user_movie_interaction` (D2 rank #1) once the multivariate model controlled for component feature redundancy. Proposed concrete D4 improvements: switching to Gradient Boosted Trees to capture nonlinear genre × age interactions, adding one-hot encoding of `primary_genre` via `StringIndexer + OneHotEncoder`, and testing one-hot encoding of `Age` buckets for non-monotone demographic effects. Learned how logistic regression's single global weight per feature prevents it from capturing conditional relationships — the fundamental limitation motivating the D4 model upgrade.